# Stage 3 - IPN Hand feature extraction (Kaggle GPU)

Extracts the frozen VideoMAEv2-Base RGB stream into the canonical `.npy` cache, which is
then checksummed and shipped back. CLAUDE.md section 4: extract once, train anywhere.

**Before running**

1. Settings -> Accelerator -> **GPU T4 x2** (or P100).
2. Settings -> **Internet: On** (needed for pip and the checkpoint download).
3. Attach your `ipn-hand-frames` dataset: the 5 `.tgz` files plus the `annotations/` folder.
4. Set `SHARD` below. The 12-hour session cap means a long extraction is split across runs;
   each run skips whatever is already cached, so re-running is always safe.

**Why the repo is cloned at a pinned commit.** Hard Rule 4 wants every number traceable to a
config, a seed, a split file and a git SHA. A notebook running pasted code has none of those.

In [ ]:
# --- what this run is ------------------------------------------------------------
REPO = 'https://github.com/asthaberi/sos-oad.git'
COMMIT = 'main'        # pin to a SHA for a run whose number you intend to report
SHARD = '0/4'          # i/n -- which slice of the 200 videos this session handles
FEATURES = 'videomaev2'

DATASET_DIR = '/kaggle/input/ipn-hand-frames'   # adjust to your dataset's slug
WORK = '/kaggle/working/sos-oad'

## 1. Code, pinned

In [ ]:
!git clone -q $REPO $WORK
%cd $WORK
!git checkout -q $COMMIT
!git log --oneline -1
# Must print nothing. A dirty tree is not a reproducible run.
!git status --porcelain | head

In [ ]:
# torch and CUDA are already on the Kaggle image; install only what the repo adds.
!pip install -q transformers==4.46.3 omegaconf==2.3.0 'Pillow>=11.0.0'
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 2. Rebuild the canonical dataset

`annotations.csv` and `meta.yaml` are *derived*, so they are regenerated from the raw
annotations rather than copied. That means the adapter's Stage 1 checks run here too.

The frozen split is the one thing never regenerated: it is committed in the repo and applied
as-is. `--apply-frozen` refuses it if the subject digest no longer matches the dataset.

In [ ]:
import os, shutil, glob
# The annotation text files sit at the dataset root, not in a subfolder: a subfolder
# would have forced `kaggle datasets create --dir-mode`, which tars or zips it, and what
# then lands in /kaggle/input becomes an avoidable unknown.
os.makedirs('raw/IPN_Hand/annotations', exist_ok=True)
for pattern in ('*.txt', '*.csv', '*.xlsx'):
    for path in glob.glob(f'{DATASET_DIR}/{pattern}'):
        shutil.copy(path, 'raw/IPN_Hand/annotations/')
print(sorted(os.listdir('raw/IPN_Hand/annotations')))
!python scripts/prepare_ipn_hand.py --raw raw/IPN_Hand --set dataset=ipn_hand 2>&1 | tail -25

In [ ]:
!python scripts/make_splits.py --set dataset=ipn_hand split=ipn_official \
    --apply-frozen 2>&1 | tail -20

## 3. Untar only this shard's frames

Session disk is limited, so the archives are expanded selectively. Shards are contiguous over
the sorted video list, which is why a shard's videos cluster within a few archives.

In [ ]:
import sys, tarfile, pathlib
sys.path.insert(0, '.')
from src.data import canonical as C
from scripts.extract_features import shard_of

wanted = set(shard_of(C.CanonicalDataset('data/ipn_hand').video_ids, SHARD))
print(len(wanted), 'videos in shard', SHARD)

os.makedirs('raw/IPN_Hand/frames', exist_ok=True)
for archive in sorted(pathlib.Path(DATASET_DIR).glob('frames*.tgz')):
    # 'r|gz' streams the archive instead of seeking, which is far faster here.
    with tarfile.open(archive, 'r|gz') as tar:
        kept = 0
        for member in tar:
            parts = member.name.split('/')
            if len(parts) > 1 and parts[1] in wanted:
                tar.extract(member, 'raw/IPN_Hand')
                kept += 1
    print(archive.name, '->', kept, 'files')

!du -sh raw/IPN_Hand/frames

## 4. Timing probe

Two videos before committing the session. Multiply the reported fps out to this shard's frame
count and check it fits inside the session cap; if it does not, raise `n` in `SHARD`.

For reference, JPEG decode alone runs at ~290 fps single-threaded on the author's laptop, so
decode rather than the GPU is the thing to watch.

In [ ]:
!python scripts/extract_features.py --set dataset=ipn_hand split=ipn_official \
    features=$FEATURES device=cuda --shard $SHARD --limit 2 2>&1 | tail -10

## 5. Extract the shard

In [ ]:
!python scripts/extract_features.py --set dataset=ipn_hand split=ipn_official \
    features=$FEATURES device=cuda --shard $SHARD 2>&1 | tail -30

## 6. Verify, then package the cache

Every array is checksummed into `meta.yaml` as it is written. Re-checking here means that a
later failure points at the transfer rather than at the extraction.

In [ ]:
!python scripts/extract_features.py --verify --set dataset=ipn_hand features=$FEATURES

In [ ]:
# Ship features + meta.yaml + the run directory. The frames stay behind: 20 GB and
# reproducible from the archives.
out = '/kaggle/working/cache'
os.makedirs(out, exist_ok=True)
shutil.copytree('data/ipn_hand/features', f'{out}/features', dirs_exist_ok=True)
shutil.copy('data/ipn_hand/meta.yaml', f'{out}/meta.yaml')
shutil.copytree('runs', f'{out}/runs', dirs_exist_ok=True)   # Hard Rule 4 provenance
shutil.make_archive('/kaggle/working/ipn_features_' + SHARD.replace('/', 'of'), 'zip', out)
!ls -lh /kaggle/working/*.zip

### Merging shards

Each shard's `meta.yaml` records only its own checksums. After downloading every shard, unzip
them all into `data/ipn_hand/` in order - `record_cache` merges rather than overwrites, so the
`meta.yaml` from the **last** shard unzipped carries every digest. Then, locally:

```powershell
python scripts/extract_features.py --verify --set dataset=ipn_hand features=videomaev2
```

`STAGE 3 GATE: PASS` means all 200 videos are cached and every checksum matches.